In [23]:
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

notebook_name = "Gold"
run_start = datetime.utcnow()
logger.info(f"[START] {notebook_name} — {run_start.strftime('%Y-%m-%d %H:%M:%S')} UTC")

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 25, Finished, Available, Finished, False)

INFO:__main__:[START] Gold — 2026-04-24 03:19:37 UTC


In [24]:
# Run this at the top of each notebook to see what tables it touches
# Paste the output for all 4 notebooks

spark.sql("SHOW TABLES").show(50, truncate=False)

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 26, Finished, Available, Finished, False)

+-----------------------------------------------------------------+-----------------------------+-----------+
|namespace                                                        |tableName                    |isTemporary|
+-----------------------------------------------------------------+-----------------------------+-----------+
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|bronze_vehicle_images        |false      |
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|complaints                   |false      |
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|gold_brand_reliability       |false      |
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|gold_component_failures      |false      |
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|gold_dim_failure_category    |false      |
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|gold_dim_make                |false      |
|Vehicle_r

In [25]:
from pyspark.sql.functions import (
    col, sum as spark_sum, avg, count, 
    round as spark_round, max as spark_max,
    min as spark_min, current_timestamp, lit,to_date,date_format
)

# read the delta table


gold_complaints = spark.read.format("delta").table("silver_complaints")
gold_recalls = spark.read.format("delta").table("silver_recalls")
gold_logos = spark.read.format("delta").table("silver_logos")
gold_maintenance_data = spark.read.format("delta").table("silver_maintenance_data")
gold_vehicle_images = spark.read.format("delta").table("silver_vehicle_images")

print(f"gold complaints row count: {gold_complaints.count()}")
print(f"gold recalls row count: {gold_recalls.count()}")
print(f"gold logos row count: {gold_logos.count()}")
print(f"gold maintenance data row count: {gold_maintenance_data.count()}")
print(f"gold vehicle images row count: {gold_vehicle_images.count()}")





StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 27, Finished, Available, Finished, False)

gold complaints row count: 178648
gold recalls row count: 8061
gold logos row count: 27
gold maintenance data row count: 2662
gold vehicle images row count: 77


In [26]:
from pyspark.sql.functions import col, count, countDistinct, avg, min as min_, max as max_

#  Aggregate complaints per brand 
brand_complaints = gold_complaints.groupBy("vehicle_make") \
    .agg(count("*").alias("total_complaints"))




# Aggregate recalls per brand 

brand_recalls = gold_recalls.groupBy("vehicle_make") \
    .agg(countDistinct("recall_id").alias("total_recalls"))

#  Aggregate avg repair cost per brand 
brand_cost = gold_maintenance_data.groupBy("vehicle_make") \
    .agg(avg("avg_annual_repair_cost").alias("avg_repair_cost"))  # fixed: singular

# Full outer join —
gold_brand = brand_complaints \
    .join(brand_recalls, on="vehicle_make", how="full") \
    .join(brand_cost,    on="vehicle_make", how="full")

#Fill missing counts with 0 — missing = no record = fair assumption 
gold_brand = gold_brand.fillna(0, subset=["total_complaints", "total_recalls"])

# Fill missing cost with dataset average — neutral, avoids rewarding unknown
# brands with cheapest cost score
avg_cost_fill = gold_brand.select(avg("avg_repair_cost")).collect()[0][0]
gold_brand = gold_brand.fillna(avg_cost_fill, subset=["avg_repair_cost"])

display(gold_brand)

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d6239c84-4e61-44b3-affb-dcc677bfafca)

In [27]:
from pyspark.sql.functions import col, lit, when, min as min_, max as max_, count, round as round_

#  Brand Reliability 
# Complaints = 40% weight
# Recalls    = 30% weight
# Repair Cost= 30% weight




#  Single aggregation for all normalization ranges 
agg_vals = gold_brand.agg(
    min_("total_complaints").alias("min_complaints"),
    max_("total_complaints").alias("max_complaints"),
    min_("total_recalls").alias("min_recalls"),
    max_("total_recalls").alias("max_recalls"),
    min_("avg_repair_cost").alias("min_cost"),
    max_("avg_repair_cost").alias("max_cost")
).collect()[0]



# Division by zero guards — if all brands have same value, range = 0
range_c    = (agg_vals["max_complaints"] - agg_vals["min_complaints"]) or 1
range_r    = (agg_vals["max_recalls"]    - agg_vals["min_recalls"])    or 1
range_cost = (agg_vals["max_cost"]       - agg_vals["min_cost"])       or 1



#  Normalize complaints to 0–100 
# 0 = fewest complaints (best), 100 = most complaints (worst)
gold_brand = gold_brand.withColumn(
    "complaints_norm",
    ((col("total_complaints") - lit(agg_vals["min_complaints"])) / lit(range_c)) * 100
)




# ── 3. Normalize recalls to 0–100 ─────────────────────────────────────────
# 0 = fewest recalls (best), 100 = most recalls (worst)
gold_brand = gold_brand.withColumn(
    "recalls_norm",
    ((col("total_recalls") - lit(agg_vals["min_recalls"])) / lit(range_r)) * 100
)



# ── 4. Normalize Maintenance repair data cost to 0–100 
# 0 = cheapest (best), 100 = most expensive (worst)
gold_brand = gold_brand.withColumn(
    "cost_norm",
    ((col("avg_repair_cost") - lit(agg_vals["min_cost"])) / lit(range_cost)) * 100
)

# Weighted reliability score 
# All three metrics are negatives so i inverted : higher score = more reliable
# Score range: 0 (worst) – 100 (best)
gold_brand = gold_brand.withColumn(
    "reliability_score",
    round_(
        100 - (
            col("complaints_norm") * 0.4 +
            col("recalls_norm")    * 0.3 +
            col("cost_norm")       * 0.3
        ),
        2
    )
)

# Score colour bands 
# Green  ≥ 70  →  reliable
# Amber  ≥ 40  →  average
# Red    < 40  →  poor
gold_brand = gold_brand.withColumn(
    "score_colour",
    when(col("reliability_score") >= 70, "#2ECC71")
    .when(col("reliability_score") >= 40, "#F39C12")
    .otherwise("#E74C3C")
    
)

# Sanity checks 
print(f"Total brands: {gold_brand.count()}")

print("\nScore range (should be 0–100):")
gold_brand.agg(
    min_("reliability_score").alias("min_score"),
    max_("reliability_score").alias("max_score")
).show()

print("\nNull check (should all be 0):")
from pyspark.sql.functions import count as count_, when as when_
gold_brand.select([
    count_(when_(col(c).isNull(), c)).alias(c)
    for c in ["complaints_norm", "recalls_norm", "cost_norm", "reliability_score"]
]).show()

print("\nColour band distribution:")
gold_brand.groupBy("score_colour").count().orderBy("score_colour").show()

display(gold_brand)

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 29, Finished, Available, Finished, False)

Total brands: 27

Score range (should be 0–100):
+---------+---------+
|min_score|max_score|
+---------+---------+
|    17.41|    100.0|
+---------+---------+


Null check (should all be 0):
+---------------+------------+---------+-----------------+
|complaints_norm|recalls_norm|cost_norm|reliability_score|
+---------------+------------+---------+-----------------+
|              0|           0|        0|                0|
+---------------+------------+---------+-----------------+


Colour band distribution:
+------------+-----+
|score_colour|count|
+------------+-----+
|     #2ECC71|   16|
|     #E74C3C|    1|
|     #F39C12|   10|
+------------+-----+



SynapseWidget(Synapse.DataFrame, 25dce584-cf09-49df-a362-adfd74fe30c4)

In [28]:
from pyspark.sql.functions import col, count, countDistinct, avg, min, max, lit, when

 #  Aggregated metrics per model
model_complaints = gold_complaints.groupBy("vehicle_make","vehicle_model","model_year") \
    .agg(count("*").alias("complaint_count"))

model_recalls = gold_recalls.groupBy("vehicle_make","vehicle_model","model_year") \
    .agg(countDistinct("recall_id").alias("recall_count"))

model_costs = gold_maintenance_data.groupBy("vehicle_make", "vehicle_model", "model_year") \
    .agg(avg("avg_annual_repair_cost").alias("avg_repair_cost"))



#  Join
gold_model = model_complaints \
    .join(model_recalls, ["vehicle_make","vehicle_model","model_year"], "left") \
    .join(model_costs, ["vehicle_make","vehicle_model","model_year"], "left")



#  Smart fillna
gold_model = gold_model.fillna(0, subset=["complaint_count", "recall_count"])



avg_cost_fill = gold_model.select(avg("avg_repair_cost")).collect()[0][0]
gold_model = gold_model.fillna(avg_cost_fill, subset=["avg_repair_cost"])



# Normalization ranges with division by zero guard
stats = gold_model.select(
    min("complaint_count").alias("min_c"), max("complaint_count").alias("max_c"),
    min("recall_count").alias("min_r"), max("recall_count").alias("max_r"),
    min("avg_repair_cost").alias("min_cost"), max("avg_repair_cost").alias("max_cost")
).collect()[0]

range_c    = stats["max_c"]    - stats["min_c"]    or 1
range_r    = stats["max_r"]    - stats["min_r"]    or 1
range_cost = stats["max_cost"] - stats["min_cost"] or 1



# Reliability score
gold_model = gold_model.withColumn(
    "reliability_score",
    100 - (
        ((col("complaint_count") - lit(stats["min_c"]))    / lit(range_c))    * 40 +
        ((col("recall_count")    - lit(stats["min_r"]))    / lit(range_r))    * 30 +
        ((col("avg_repair_cost") - lit(stats["min_cost"])) / lit(range_cost)) * 30
    )
)



#  Display
gold_model.select(
    "vehicle_make","vehicle_model","model_year",
    "complaint_count","recall_count","avg_repair_cost","reliability_score"
).orderBy(col("reliability_score").desc())

display(gold_model)

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b032265b-8d55-4529-b303-8d12c0e35d3c)

In [29]:
# Failure Analysis

from pyspark.sql.functions import when, upper, trim, sum as sum_

component_clean = gold_complaints.withColumn(
    "component",
    trim(upper(col("component")))
)

# Components Standardized

component_clean = component_clean.withColumn(
    "component",
    when(col("component").isin("ENG","ENGINE BLOCK"), "ENGINE")
    .when(col("component").isin("TRANS","TRANSMISSION"), "TRANSMISSION")
    .otherwise(col("component"))
)



component_failures = component_clean.groupBy("component") \
    .agg(count("*").alias("failure_count"))

# Calculate percentage of total failures
total_failures = component_failures.agg(sum_("failure_count")).collect()[0][0]

component_failures = component_failures.withColumn(
    "failure_percentage",
    (col("failure_count") / total_failures) * 100
)

component_failures.orderBy(col("failure_percentage").desc()).show(10)

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 31, Finished, Available, Finished, False)

+--------------------+-------------+------------------+
|           component|failure_count|failure_percentage|
+--------------------+-------------+------------------+
|              ENGINE|        21644|12.115444897228068|
|    UNKNOWN OR OTHER|        15971|  8.93992655949129|
|         POWER TRAIN|        15926| 8.914737360619766|
|   ELECTRICAL SYSTEM|        11961|  6.69528906005105|
|            STEERING|         8054|4.5083068380278535|
|    VISIBILITY/WIPER|         6019| 3.369195289060051|
|           STRUCTURE|         5666| 3.171600017912319|
|      SERVICE BRAKES|         5422|3.0350185840311674|
|            AIR BAGS|         4825| 2.700841879002284|
|FUEL/PROPULSION S...|         4424|2.4763781290582596|
+--------------------+-------------+------------------+
only showing top 10 rows



In [30]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim, upper, explode, split, lit
from pyspark.sql.window import Window

# Load silver complaints 
silver_complaints = spark.read.format("delta").load("Tables/dbo/silver_complaints")

#  Category mapping — NHTSA raw → clean display label 
category_map = {
    "ENGINE":                        "Engine",
    "ENGINE AND ENGINE COOLING":     "Engine",
    "POWER TRAIN":                   "Powertrain",
    "ELECTRICAL SYSTEM":             "Electrical",
    "STEERING":                      "Steering",
    "SERVICE BRAKES":                "Brakes",
    "AIR BAGS":                      "Air Bags",
    "FUEL/PROPULSION SYSTEM":        "Fuel System",
    "FUEL SYSTEM, GASOLINE":         "Fuel System",
    "FORWARD COLLISION AVOIDANCE":   "Safety Systems",
    "VEHICLE SPEED CONTROL":         "Safety Systems",
    "VISIBILITY/WIPER":              "Visibility",
    "VISIBILITY":                    "Visibility",
    "EXTERIOR LIGHTING":             "Lighting",
    "STRUCTURE":                     "Structure",
    "UNKNOWN OR OTHER":              "Other",
}

mapping_expr = F.create_map(
    *[item for pair in
      [(F.lit(k), F.lit(v)) for k, v in category_map.items()]
      for item in pair]
)

#  Explode comma-separated components into individual rows 
exploded = (
    silver_complaints
    .select(
        "complaint_id",
        "vehicle_make",
        "vehicle_model",
        "model_year",
        "component"
    )
    .withColumn(
        "component_raw",
        explode(split(col("component"), ","))
    )
    .withColumn(
        "component_raw",
        trim(upper(col("component_raw")))
    )
)

#  Map to clean categories 
#       Log unmapped rows before filtering so nothing is silently lost
mapped_with_other = (
    exploded
    .withColumn(
        "component_category",
        F.coalesce(
            mapping_expr[col("component_raw")],
            F.lit("Other")
        )
    )
    .filter(col("vehicle_make").isNotNull())
)



# Audit — how many rows are being dropped as unmapped
other_count = mapped_with_other.filter(col("component_category") == "Other").count()
total_exploded = mapped_with_other.count()
print(f"Unmapped rows dropped as Other: {other_count} / {total_exploded} ({round(other_count/total_exploded*100, 2)}%)")

# Now filter out Other
mapped = mapped_with_other.filter(col("component_category") != "Other")

#  Aggregate to brand + category level 
gold_failure = (
    mapped
    .groupBy("vehicle_make", "component_category")
    .agg(
        F.count("complaint_id").alias("failure_count")
    )
)

#  Per-brand percentage using window function 
#       "Of this brand's total failures, what % were this component?"
#       Each brand's categories will sum to 100% — correct for Power BI filtering

brand_window = Window.partitionBy("vehicle_make")

gold_failure = (
    gold_failure
    .withColumn(
        "brand_total_failures",
        F.sum("failure_count").over(brand_window)
    )
    .withColumn(
        "failure_percentage",
        F.round((col("failure_count") / col("brand_total_failures")) * 100, 4)
    )
    .orderBy("vehicle_make", col("failure_count").desc())
)

#  Preview before writing 
print(f"Total rows: {gold_failure.count()}")
print(f"Distinct brands: {gold_failure.select('vehicle_make').distinct().count()}")
print(f"Distinct categories: {gold_failure.select('component_category').distinct().count()}")

# Sanity check — every brand's percentages should sum to 100
print("\nPercentage sum per brand (should all be 100.0):")
gold_failure.groupBy("vehicle_make") \
    .agg(F.round(F.sum("failure_percentage"), 2).alias("pct_sum")) \
    .orderBy("vehicle_make") \
    .show(20, truncate=False)

gold_failure.show(30, truncate=False)

#  Write to Gold 
gold_failure.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_failure_category")

print("gold_dim_failure_category written successfully")

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 32, Finished, Available, Finished, False)

Unmapped rows dropped as Other: 58223 / 255701 (22.77%)
Total rows: 275
Distinct brands: 25
Distinct categories: 11

Percentage sum per brand (should all be 100.0):
+-------------+-------+
|vehicle_make |pct_sum|
+-------------+-------+
|Acura        |100.0  |
|Audi         |100.0  |
|Bmw          |100.0  |
|Buick        |100.0  |
|Cadillac     |100.0  |
|Chevrolet    |100.0  |
|Chrysler     |100.0  |
|Dodge        |100.0  |
|Ford         |100.0  |
|Gmc          |100.0  |
|Honda        |100.0  |
|Hyundai      |100.0  |
|Infiniti     |100.0  |
|Jaguar       |100.0  |
|Jeep         |100.0  |
|Kia          |100.0  |
|Lexus        |100.0  |
|Lincoln      |100.0  |
|Mazda        |100.0  |
|Mercedes-Benz|100.0  |
+-------------+-------+
only showing top 20 rows

+------------+------------------+-------------+--------------------+------------------+
|vehicle_make|component_category|failure_count|brand_total_failures|failure_percentage|
+------------+------------------+-------------+----------

In [31]:
df = spark.read.format("delta").load("Tables/dbo/gold_dim_failure_category")

print("Total rows:", df.count())
print("\nSample components:")
df.orderBy("failure_count", ascending=False).show(20, truncate=False)

print("\nDistinct component count:")
print(df.select("component_category").distinct().count())

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 33, Finished, Available, Finished, False)

Total rows: 275

Sample components:
+------------+------------------+-------------+--------------------+------------------+
|vehicle_make|component_category|failure_count|brand_total_failures|failure_percentage|
+------------+------------------+-------------+--------------------+------------------+
|Ford        |Engine            |9085         |26563               |34.2017           |
|Ford        |Powertrain        |5851         |26563               |22.0269           |
|Hyundai     |Engine            |5613         |17414               |32.2327           |
|Jeep        |Electrical        |5186         |22607               |22.9398           |
|Chevrolet   |Engine            |5022         |20578               |24.4047           |
|Kia         |Engine            |4730         |12896               |36.678            |
|Honda       |Engine            |4603         |23418               |19.6558           |
|Jeep        |Engine            |4536         |22607               |20.0646         

In [32]:
from pyspark.sql.functions import avg, count, col, round as round_

#  Aggregate to make/model/year grain 
# One row per make/model/year already exists in source,
# but i aggregate defensively in case of duplicates after joins

cost_table = gold_maintenance_data.groupBy("vehicle_make", "vehicle_model", "model_year") \
    .agg(
        avg("avg_annual_repair_cost").alias("avg_annual_repair_cost"),
        avg("maintenance_frequency").alias("avg_maintenance_frequency"),
        count("*").alias("record_count")   # for audit — should always be 1
    )

# Sanity check — flag any duplicates 
from pyspark.sql.functions import col
dupes = cost_table.filter(col("record_count") > 1)
print(f"Duplicate make/model/year rows: {dupes.count()}")

# Correct annual cost formula 

cost_table = cost_table.withColumn(
    "expected_annual_cost",
    round_(col("avg_annual_repair_cost") * col("avg_maintenance_frequency"), 2)
)

#  Display 
cost_table.select(
    "vehicle_make", "vehicle_model", "model_year",
    "avg_annual_repair_cost", "avg_maintenance_frequency", "expected_annual_cost"
).orderBy("vehicle_make", "vehicle_model", "model_year") \
 .show(20, truncate=False)

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 34, Finished, Available, Finished, False)

Duplicate make/model/year rows: 0
+------------+-------------+----------+----------------------+-------------------------+--------------------+
|vehicle_make|vehicle_model|model_year|avg_annual_repair_cost|avg_maintenance_frequency|expected_annual_cost|
+------------+-------------+----------+----------------------+-------------------------+--------------------+
|Acura       |Adx          |2016      |473.0                 |0.4                      |189.2               |
|Acura       |Adx          |2017      |485.0                 |0.4                      |194.0               |
|Acura       |Adx          |2018      |497.0                 |0.4                      |198.8               |
|Acura       |Adx          |2019      |510.0                 |0.4                      |204.0               |
|Acura       |Adx          |2020      |538.0                 |0.4                      |215.2               |
|Acura       |Adx          |2021      |568.0                 |0.4                     

In [33]:
from pyspark.sql.functions import col, avg, min as min_, max as max_, round as round_

# Collapse cost_table to make/model level 

cost_table_model_level = cost_table.groupBy("vehicle_make", "vehicle_model") \
    .agg(
        avg("avg_annual_repair_cost").alias("avg_annual_repair_cost"),
        avg("avg_maintenance_frequency").alias("avg_maintenance_frequency"),
        avg("expected_annual_cost").alias("annual_cost")
    )

# Join onto gold_model (make/model/year grain) 
recommendations = gold_model.join(
    cost_table_model_level, on=["vehicle_make", "vehicle_model"], how="left"
)

# Audit — check for unmatched models (NULL annual_cost after join) 
unmatched = recommendations.filter(col("annual_cost").isNull())
print(f"Models with no cost data after join: {unmatched.count()}")
unmatched.select("vehicle_make", "vehicle_model", "model_year").show(20, truncate=False)

#  Normalize annual cost using joined population 

cost_stats = recommendations.agg(
    min_("annual_cost").alias("min_cost"),
    max_("annual_cost").alias("max_cost")
).collect()[0]

range_cost = (cost_stats["max_cost"] - cost_stats["min_cost"]) or 1  # div by zero guard

recommendations = recommendations.withColumn(
    "norm_cost",
    (col("annual_cost") - cost_stats["min_cost"]) / range_cost
)

#  Recommendation score 
# reliability_score → 0–100, higher is better, weight 70%
# norm_cost         → 0–1,   lower is better,  weight 30% (inverted to 0–100)
#
# NB: repair cost already contributes 30% inside reliability_score,
# so effective cost influence = (0.3 × 0.7) + 0.30 = 51% of final score.
# This is intentional — document if surfacing scores externally.

recommendations = recommendations.withColumn(
    "recommendation_score",
    round_(
        col("reliability_score") * 0.7 + (100 - col("norm_cost") * 100) * 0.3,
        2
    )
)

#  Sanity checks 
print("\nScore range (should be 0–100):")
recommendations.agg(
    min_("recommendation_score").alias("min_score"),
    max_("recommendation_score").alias("max_score")
).show()

print("Null scores (should be 0):")
recommendations.filter(col("recommendation_score").isNull()).count()

#  Display 
display(
    recommendations.select(
        "vehicle_make",
        "vehicle_model",
        "model_year",
        "reliability_score",
        "annual_cost",
        "norm_cost",
        "recommendation_score"
    ).orderBy(col("recommendation_score").desc())
)

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 35, Finished, Available, Finished, False)

Models with no cost data after join: 0
+------------+-------------+----------+
|vehicle_make|vehicle_model|model_year|
+------------+-------------+----------+
+------------+-------------+----------+


Score range (should be 0–100):
+---------+---------+
|min_score|max_score|
+---------+---------+
|    52.77|    99.46|
+---------+---------+

Null scores (should be 0):


SynapseWidget(Synapse.DataFrame, 6ebce83c-2a2e-48b0-99eb-87dd3ad6e1a9)

In [34]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

tables = [
    "gold_brand_reliability",
    "gold_model_reliability",
    "gold_component_failures",
    "gold_total_cost",
    "gold_recommendations"
]

for t in tables:
    print(f"\n{'='*50}")
    print(f"TABLE: {t}")
    print('='*50)
    df = spark.read.table(t)
    df.printSchema()
    print(f"Row count: {df.count()}")
    df.show(2, truncate=False)

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 36, Finished, Available, Finished, False)


TABLE: gold_brand_reliability
root
 |-- vehicle_make: string (nullable = true)
 |-- total_complaints: long (nullable = true)
 |-- total_recalls: long (nullable = true)
 |-- avg_repair_cost: double (nullable = true)
 |-- complaints_norm: double (nullable = true)
 |-- recalls_norm: double (nullable = true)
 |-- cost_norm: double (nullable = true)
 |-- reliability_score: double (nullable = true)
 |-- score_colour: string (nullable = true)

Row count: 27
+------------+----------------+-------------+-----------------+------------------+------------------+------------------+-----------------+------------+
|vehicle_make|total_complaints|total_recalls|avg_repair_cost  |complaints_norm   |recalls_norm      |cost_norm         |reliability_score|score_colour|
+------------+----------------+-------------+-----------------+------------------+------------------+------------------+-----------------+------------+
|Acura       |3385            |35           |568.3545454545455|13.044315992292871|10.091

In [35]:
# Fact Vehicle Reliabillity

from pyspark.sql.functions import col, avg, min as min_, max as max_, round as round_

#  Collapse cost_table to make/model grain 
cost_table_model_level = cost_table.groupBy("vehicle_make", "vehicle_model") \
    .agg(
        avg("avg_annual_repair_cost").alias("avg_annual_repair_cost"),
        avg("avg_maintenance_frequency").alias("avg_maintenance_frequency"),
        avg("expected_annual_cost").alias("annual_cost")
    )

#  Join onto gold_model 
fact_vehicle_reliability = gold_model.join(
    cost_table_model_level, on=["vehicle_make", "vehicle_model"], how="left"
)

#  Norm cost stats drawn from this table's population 
cost_stats = fact_vehicle_reliability.agg(
    min_("annual_cost").alias("min_cost"),
    max_("annual_cost").alias("max_cost")
).collect()[0]

range_cost = (cost_stats["max_cost"] - cost_stats["min_cost"]) or 1

#  Scores 
fact_vehicle_reliability = fact_vehicle_reliability \
    .withColumn(
        "norm_cost",
        (col("annual_cost") - cost_stats["min_cost"]) / range_cost
    ) \
    .withColumn(
        "recommendation_score",
        round_(
            col("reliability_score") * 0.7 + (100 - col("norm_cost") * 100) * 0.3,
            2
        )
    )

#  Select final columns 
fact_vehicle_reliability = fact_vehicle_reliability.select(
    "vehicle_make",
    "vehicle_model",
    "model_year",
    "complaint_count",
    "recall_count",
    "avg_repair_cost",           
    "avg_maintenance_frequency", 
    "annual_cost",
    "reliability_score",
    "recommendation_score"
)

#  Sanity checks before writing 
print(f"Total rows: {fact_vehicle_reliability.count()}")
print(f"Null reliability scores: {fact_vehicle_reliability.filter(col('reliability_score').isNull()).count()}")
print(f"Null recommendation scores: {fact_vehicle_reliability.filter(col('recommendation_score').isNull()).count()}")

fact_vehicle_reliability.agg(
    min_("reliability_score").alias("min_reliability"),
    max_("reliability_score").alias("max_reliability"),
    min_("recommendation_score").alias("min_recommendation"),
    max_("recommendation_score").alias("max_recommendation")
).show()

# Write to Gold 
fact_vehicle_reliability.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_fact_vehicle_reliability")

print("gold_fact_vehicle_reliability written successfully")

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 37, Finished, Available, Finished, False)

Total rows: 1314
Null reliability scores: 0
Null recommendation scores: 0
+-----------------+----------------+------------------+------------------+
|  min_reliability| max_reliability|min_recommendation|max_recommendation|
+-----------------+----------------+------------------+------------------+
|40.93149426522334|99.6473716726208|             52.77|             99.46|
+-----------------+----------------+------------------+------------------+

gold_fact_vehicle_reliability written successfully


In [36]:
from pyspark.sql.functions import col, count, when, min as min_, max as max_

#  Load Gold table 
df = spark.read.format("delta").load("Tables/dbo/gold_fact_vehicle_reliability")

print(f"Total rows: {df.count()}")
print(f"Distinct makes: {df.select('vehicle_make').distinct().count()}")
print(f"Distinct models: {df.select('vehicle_model').distinct().count()}")

#  audit 

print("\n=== NULL counts (should all be 0) ===")
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in [
        "reliability_score",
        "recommendation_score",
        "annual_cost",
        "complaint_count",
        "recall_count"
    ]
]).show()

# Score range check — both scores must stay within 0–100 
print("=== Score range (min and max should be within 0–100) ===")
df.agg(
    min_("reliability_score").alias("min_reliability"),
    max_("reliability_score").alias("max_reliability"),
    min_("recommendation_score").alias("min_recommendation"),
    max_("recommendation_score").alias("max_recommendation")
).show()

#  Score distribution 
# Median should sit in 40–70 range
# If 25%–75% spread is very narrow (e.g. 48–52), normalization is too flat
print("=== Reliability score distribution ===")
df.select("reliability_score") \
    .summary("min", "25%", "50%", "75%", "max") \
    .show()

print("Recommendation score distribution ")
df.select("recommendation_score") \
    .summary("min", "25%", "50%", "75%", "max") \
    .show()



StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 38, Finished, Available, Finished, False)

Total rows: 1314
Distinct makes: 25
Distinct models: 197

=== NULL counts (should all be 0) ===
+-----------------+--------------------+-----------+---------------+------------+
|reliability_score|recommendation_score|annual_cost|complaint_count|recall_count|
+-----------------+--------------------+-----------+---------------+------------+
|                0|                   0|          0|              0|           0|
+-----------------+--------------------+-----------+---------------+------------+

=== Score range (min and max should be within 0–100) ===
+-----------------+----------------+------------------+------------------+
|  min_reliability| max_reliability|min_recommendation|max_recommendation|
+-----------------+----------------+------------------+------------------+
|40.93149426522334|99.6473716726208|             52.77|             99.46|
+-----------------+----------------+------------------+------------------+

=== Reliability score distribution ===
+-------+------------

In [37]:
from pyspark.sql.functions import col, length, row_number
from pyspark.sql.window import Window



# gold_dim_vehicle 
# dropDuplicates keeps one URL per make/model/year

w = Window.partitionBy("vehicle_make", "vehicle_model", "model_year") \
          .orderBy(length(col("url")).desc())

dim_vehicle = gold_vehicle_images \
    .select("vehicle_make", "vehicle_model", "model_year", "url") \
    .withColumn("rn", row_number().over(w)) \
    .filter(col("rn") == 1) \
    .drop("rn")

# Sanity check
print(f"dim_vehicle rows: {dim_vehicle.count()}")
print(f"Distinct keys: {dim_vehicle.select('vehicle_make','vehicle_model','model_year').distinct().count()}")
print("(Both numbers should match — confirms no duplicates)")

dim_vehicle.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_vehicle")

print("gold_dim_vehicle written successfully")



StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 39, Finished, Available, Finished, False)

dim_vehicle rows: 77
Distinct keys: 77
(Both numbers should match — confirms no duplicates)
gold_dim_vehicle written successfully


In [38]:
from pyspark.sql.functions import col, length, row_number
from pyspark.sql.window import Window

# gold_dim_make 
# Norm columns removed — internal scoring intermediates, not brand attributes

dim_make = gold_brand.select(
    "vehicle_make",
    "total_complaints",
    "total_recalls",
    "avg_repair_cost",
    "reliability_score",
    "score_colour",
    "complaints_norm",
    "recalls_norm"
)

# Sanity check
print(f"\ndim_make rows: {dim_make.count()}")
print(f"Distinct makes: {dim_make.select('vehicle_make').distinct().count()}")
print("(Both numbers should match — one row per brand)")

dim_make.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_make")

print("gold_dim_make written successfully")

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 40, Finished, Available, Finished, False)


dim_make rows: 27
Distinct makes: 27
(Both numbers should match — one row per brand)
gold_dim_make written successfully


In [39]:
display(dim_make)

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 41, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fa4fd574-a052-4fd7-8a87-79a863dde21d)

In [40]:
df = spark.read.format("delta").load("Tables/dbo/gold_fact_vehicle_reliability")
df.select(
    "vehicle_make", "vehicle_model", "model_year",
    "reliability_score", "recommendation_score", "annual_cost"
).orderBy("reliability_score", ascending=False).show(10, truncate=False)

print("Year range:", df.agg({"model_year": "min"}).collect()[0][0], "to", df.agg({"model_year": "max"}).collect()[0][0])
print("Score range:", df.agg({"reliability_score": "min"}).collect()[0][0], "to", df.agg({"reliability_score": "max"}).collect()[0][0])
print("Cost range:", df.agg({"annual_cost": "min"}).collect()[0][0], "to", df.agg({"annual_cost": "max"}).collect()[0][0])

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 42, Finished, Available, Finished, False)

+------------+-------------+----------+-----------------+--------------------+------------------+
|vehicle_make|vehicle_model|model_year|reliability_score|recommendation_score|annual_cost       |
+------------+-------------+----------+-----------------+--------------------+------------------+
|Toyota      |Yaris        |2018      |99.6473716726208 |99.46               |102.16363636363636|
|Toyota      |Yaris        |2020      |99.10033575975717|99.08               |102.16363636363636|
|Toyota      |Yaris        |2016      |98.93660855784469|98.96               |102.16363636363636|
|Toyota      |Yaris        |2017      |97.63575707110049|98.05               |102.16363636363636|
|Nissan      |Versa        |2019      |97.16550001343039|97.06               |136.79999999999998|
|Toyota      |Yaris        |2019      |97.15634585941068|97.72               |102.16363636363636|
|Toyota      |Prius        |2018      |96.99566467001532|96.88               |139.90909090909093|
|Nissan      |Versa 

In [41]:
# Save gold tables


#  Save Gold tables 
# gold_dim_failure_category is already  written 
# gold_fact_vehicle_reliability is already before written b

#  Brand reliability 
gold_brand.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_brand_reliability")
print("gold_brand_reliability written successfully")

#  Model reliability 
gold_model.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_model_reliability")
print("gold_model_reliability written successfully")

#  Total ownership cost (make/model/year grain — reference table) 
# NB: cost_table_model_level (collapsed to make/model) is the version

cost_table.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_total_cost")
print("gold_total_cost written successfully")

# Tables intentionally NOT written here 
# gold_dim_failure_category → written in failure analysis script
# gold_fact_vehicle_reliability → written in fact table script
# gold_recommendations → duplicate of gold_fact_vehicle_reliability, not

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 43, Finished, Available, Finished, False)

gold_brand_reliability written successfully
gold_model_reliability written successfully
gold_total_cost written successfully


In [42]:
# ── Wrap your existing NHTSA API call like this ──────────────────
import requests
from datetime import datetime

def fetch_nhtsa_with_retry(url, retries=3, timeout=30):
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.Timeout:
            logger.warning(f"NHTSA timeout — attempt {attempt + 1} of {retries}")
        except requests.exceptions.HTTPError as e:
            logger.error(f"NHTSA HTTP error: {e}")
            break
        except Exception as e:
            logger.error(f"NHTSA unexpected error: {e}")
            break
    logger.error("NHTSA API failed after all retries — using last known bronze data")
    return None

# Replace your existing requests.get() call with:
# data = fetch_nhtsa_with_retry("your_nhtsa_url_here")
# if data is None:
#     dbutils.notebook.exit("NHTSA_FAILED")

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 44, Finished, Available, Finished, False)

In [43]:
# ── Pipeline run completion log ──────────────────────────────────
run_end = datetime.utcnow()
duration = (run_end - run_start).seconds
logger.info(f"[END] {notebook_name} — completed in {duration}s")

# Write run log to lakehouse for monitoring
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

log_schema = StructType([
    StructField("notebook_name",  StringType(),   True),
    StructField("run_start",      TimestampType(), True),
    StructField("run_end",        TimestampType(), True),
    StructField("duration_secs",  IntegerType(),  True),
    StructField("status",         StringType(),   True),
])

log_row = [Row(
    notebook_name = notebook_name,
    run_start     = run_start,
    run_end       = run_end,
    duration_secs = duration,
    status        = "SUCCESS"
)]

log_df = spark.createDataFrame(log_row, schema=log_schema)

log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("pipeline_run_log")

print(f"[DONE] {notebook_name} — {duration}s")

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 45, Finished, Available, Finished, False)

[DONE] Gold — 94s


In [44]:
spark.read.format("delta").load("Tables/dbo/pipeline_run_log").show(truncate=False)

StatementMeta(, dd7a84bd-9ff8-4ca2-9f0f-a1dfeb91ef7c, 46, Finished, Available, Finished, False)

+----------------+--------------------------+--------------------------+-------------+-------+
|notebook_name   |run_start                 |run_end                   |duration_secs|status |
+----------------+--------------------------+--------------------------+-------------+-------+
|Bronze_ingestion|2026-04-24 02:32:21.73143 |2026-04-24 02:54:46.748086|1345         |SUCCESS|
|Silver          |2026-04-24 03:07:20.991761|2026-04-24 03:08:46.294357|85           |SUCCESS|
|Bronze          |2026-04-24 03:03:00.139947|2026-04-24 03:04:06.873314|66           |SUCCESS|
|Gold            |2026-04-24 03:10:08.721127|2026-04-24 03:14:31.685846|262          |SUCCESS|
|Gold            |2026-04-24 03:19:37.17674 |2026-04-24 03:21:11.525113|94           |SUCCESS|
|NULL            |2026-04-23 16:56:04.262779|2026-04-23 17:22:49.896034|1605         |SUCCESS|
|NULL            |2026-04-23 19:06:23.265507|2026-04-23 19:26:40.750176|1217         |SUCCESS|
|NULL            |2026-04-23 17:36:41.728831|2026-